# Data-Level Security Using ABAC (Governed Tags)

This notebook demonstrates how data-level security can be implemented using Attribute-Based Access Control (ABAC) in Unity Catalog.

In previous examples, row filters and column masks were applied directly to individual tables. While effective, that approach requires policies to be defined and managed separately for each table.

ABAC addresses this by introducing a centralized model where:

  - Data is tagged using governed attributes  
  - Policies are defined once based on those attributes  
  - Policies are applied automatically across multiple objects  

The objective is to:

  - Classify sensitive data using governed tags  
  - Define reusable policies based on those tags  

  > Note: Row filters and column masks are not supported on Dedicated (Single User) clusters as of now. Use a Shared cluster or a Serverless cluster to run this demo.

### Step 1: Set up the schema for the table 


In [0]:
%sql
USE CATALOG demo;
USE SCHEMA data_security;


### Step 2: Create the table 
A table is defined to represent sales data across multiple regions. 

In [0]:
%sql
CREATE OR REPLACE TABLE  sales_abac (
    id INT,
    region STRING,
    email STRING,
    revenue INT
);

### Step 3: Insert Sample Data 

In [0]:
%sql
INSERT INTO sales_abac VALUES
    (1, 'UK', 'john.smith@example.com', 100),
    (2, 'US', 'jane.doe@example.com', 200),
    (3, 'UK', 'jack.black@example.com', 300),
    (4, 'US', 'jill.white@example.com', 400),
    (5, 'UK', 'jim.green@example.com', 500),
    (6, 'US', 'joe.brown@example.com', 600),
    (7, 'UK', 'jane.smith@example.com', 700),
    

### Step 4: Create UDFs for row filter and column mask

(We have already created them as part of previous lecture)

1. fn_filter_region  
(This function evaluates the user’s group membership and determines whether to include a specific record in the output based on the region)

2. fn_mask_email  
(This function evaluates the user’s group membership and determines whether to return the original value or a masked version of the value)

In [0]:
%sql
CREATE OR REPLACE FUNCTION fn_filter_region (region STRING)
RETURN
    is_account_group_member('admin-sg')
    OR (is_account_group_member('uk-sg') AND region = 'UK')
    OR (is_account_group_member('us-sg') AND region = 'US');

### Step 5: Create Governed Tags

1. pii = email  
(Indicates that the data contains personally identifiable information (PII))

2. access_type = region  
(Defines the type of access control to be applied to the data, such as region-based or department-based filtering)

### Step 6: Create Policy

(Could be at catalog/ schema/ table level)

1. policy_filter_region  
2. policy_mask_email

### Step 7: Attach tags to the table 
1. sales_abac.region
2. sales_abac.email

In [0]:
%sql
SELECT * FROM sales_abac;